In [1]:
import docker
import os
import sys
import json
import subprocess
import multiprocessing

client = docker.from_env()
try:
    info = client.info()
    print(f"Docker erfolgreich verbunden. Server-Version: {info['ServerVersion']}")
except Exception as e:
    print(f"Konnte keine Verbindung zu Docker herstellen: {e}")

Docker erfolgreich verbunden. Server-Version: 29.5.2


Phase 1: Die isolierte Werkbank (Sicherheit & Infrastruktur)
Bevor die KI "denken" kann, muss der Rahmen stehen. Wir nutzen Docker als Sicherheits-Schleuse.

Docker-Container als "Sandbox": Wir erstellen ein Dockerfile, das ein minimales Python 3.10.20-Image enthält. Dieser Container bekommt keinen Internetzugriff.

Der "Sicherheits-Wächter": Wir entwickeln eine Python-Klasse, die als einzige Schnittstelle zwischen der KI und deinem Dateisystem dient. Wenn die KI eine Datei öffnen will, muss sie dieses Modul fragen. Das Modul prüft: "Liegt der Pfad im erlaubten Arbeitsordner?" Wenn nicht, blockiert es den Zugriff sofort.

In [ ]:
%run ../src/docker_py/docker_start.py

In [2]:
import os
import sys
import docker

def get_project_root() -> str:
    """Ermittelt das Stammverzeichnis über den festen Anker 'Offline_AI'."""
    if "__file__" in globals():
        current_path = os.path.abspath(os.path.dirname(__file__))
    else:
        current_path = os.path.abspath(os.getcwd())
    
    path_parts = current_path.split(os.sep)
    if "Offline_AI" in path_parts:
        ki_index = path_parts.index("Offline_AI")
        root_path = os.sep + os.path.join(*path_parts[:ki_index + 1])
        return os.path.abspath(root_path)
    return os.path.abspath(os.getcwd())

# ==============================================================================
# 1. INITIALISIERUNG & STRUKTUR
# ==============================================================================
PROJECT_ROOT = get_project_root()
DOCKER_DIR = os.path.join(PROJECT_ROOT, "docker")
os.makedirs(DOCKER_DIR, exist_ok=True)

# Automatisches Erstellen des Dockerfile im 'docker' Ordner
dockerfile_path = os.path.join(DOCKER_DIR, "Dockerfile")
if not os.path.exists(dockerfile_path):
    print(f"[INFO] Erstelle Dockerfile in {DOCKER_DIR}...")
    with open(dockerfile_path, "w") as f:
        f.write('FROM python:3.10-slim\nWORKDIR /app\nCOPY . .\nRUN pip install --no-cache-dir docker\nCMD ["python", "main.py"]')

# Sicherstellen aller Arbeitsordner
for folder in ["data", "knowledge", "privacy", "temp", "logs", "config"]:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

try:
    client = docker.from_env()
except Exception as e:
    print(f"[FEHLER] Docker-Dienst nicht erreichbar: {e}")
    sys.exit(1)

# ==============================================================================
# 2. DOCKER-LOGIK
# ==============================================================================
def start_mai_ai_container():
    image_name = "mai_ai_image:latest"
    container_name = "mai_ai"
    
    # 1. Image prüfen oder bauen
    try:
        client.images.get(image_name)
        print(f"[INFO] Image '{image_name}' lokal gefunden.")
    except docker.errors.ImageNotFound:
        print(f"[INFO] Image '{image_name}' nicht vorhanden. Starte Build aus: {DOCKER_DIR}...")
        try:
            # Build aus dem dedizierten docker-Ordner
            build_logs = client.images.build(path=DOCKER_DIR, tag=image_name, rm=True)[1]
            for log_line in build_logs:
                if 'stream' in log_line: print(log_line['stream'].strip())
            print(f"[INFO] Image erfolgreich gebaut.")
        except Exception as e:
            print(f"[FEHLER] Build-Prozess fehlgeschlagen: {e}")
            sys.exit(1)

    # 2. Aufräumen (bestehende Instanz beenden)
    try:
        container = client.containers.get(container_name)
        container.stop()
        container.remove()
    except: pass

    # 3. Starten mit Mount-Mapping (Logik: read-write für Daten/Log, read-only für Privatsphäre/Wissen)
    vol_mapping = {
        os.path.join(PROJECT_ROOT, "data"):      {'bind': '/app/data',      'mode': 'rw'},
        os.path.join(PROJECT_ROOT, "knowledge"): {'bind': '/app/knowledge', 'mode': 'ro'},
        os.path.join(PROJECT_ROOT, "privacy"):   {'bind': '/app/privacy',   'mode': 'ro'},
        os.path.join(PROJECT_ROOT, "temp"):      {'bind': '/app/temp',      'mode': 'rw'},
        os.path.join(PROJECT_ROOT, "docker"):    {'bind': '/app/docker',    'mode': 'rw'},
        os.path.join(PROJECT_ROOT, "logs"):      {'bind': '/app/logs',      'mode': 'rw'}
    }

    print(f"[INFO] Starte Container mit isolierten Mounts...")
    container = client.containers.run(
        image_name,
        name=container_name,
        detach=True,
        restart_policy={"Name": "unless-stopped"},
        volumes=vol_mapping,
        ports={'80/tcp': 8080},
        environment={"OLLAMA_HOST": "host.docker.internal"},
        mem_limit="2g",
        nano_cpus=2000000000
    )
    
    # 4. Erfolgsmeldung
    print("="*60)
    print(f"=== Mai_AI ERFOLGREICH GESTARTET ===")
    print(f" -> Container ID   : {container.short_id}")
    print(f" -> Web-Interface  : http://localhost:8080")
    print(f" -> Projektpfad    : {PROJECT_ROOT}")
    print("="*60)

if __name__ == "__main__":
    start_mai_ai_container()

[INFO] Image 'mai_ai_image:latest' lokal gefunden.
[INFO] Starte Container mit isolierten Mounts...
=== Mai_AI ERFOLGREICH GESTARTET ===
 -> Container ID   : f28bbec4957f
 -> Web-Interface  : http://localhost:8080
 -> Projektpfad    : /Users/cristallagus/Desktop/GitHub/Offline_AI


Phase 2: Das Agenten-Framework (Funktionalität)
Hier bringen wir der KI bei, ihre Werkzeuge zu benutzen, ohne dass sie "ausbüxen" kann.

Tool-Definition: Wir definieren für die KI explizite Funktionen (z. B. datei_lesen, bild_erstellen, notiz_schreiben). Die KI lernt, dass sie diese Werkzeuge aufrufen muss, anstatt "einfach so" zu handeln.

Ollama-Integration: Wir stellen sicher, dass das Modell (Codestral) lokal über eine geschlossene API angesprochen wird. Der Traffic bleibt zu 100 % auf deinem Rechner.

Phase 3: Die menschliche Kontroll-Instanz (Die Konsole)
Damit du die volle Kontrolle behältst, bauen wir eine kleine HTML-Oberfläche, die als dein persönliches Cockpit dient.

Zustimmungs-Workflow: Bei jeder kritischen Aktion (z. B. "lösche dieses Verzeichnis" oder "öffne diese Datei") erscheint in deiner Konsole eine Bestätigung. Die KI "fragt" dich, du "erlaubst" es.

Monitoring: Wir implementieren eine Anzeige, die dir zeigt, wie viel CPU und RAM die KI gerade verbraucht, damit dein System nie an seine Grenzen stößt.